# Biohub cell tracking — detector training on Colab GPU

Trains the 3D U-Net point detector on a Colab GPU (A100/L4/T4 — all far faster than
the dev Mac's MPS and, unlike the Kaggle-assigned P100, supported by PyTorch).

**Flow:** pull data + `ctrack` code from Kaggle → train → push weights back to Kaggle
as a dataset, which the submission kernel loads.

**Before running:** Runtime → Change runtime type → GPU (A100 or L4 if available).
You'll need your Kaggle API token (kaggle.com → Settings → API → Create New Token).


## 1. GPU check


In [ ]:
import torch, subprocess
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print(subprocess.getoutput('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader'))
else:
    raise SystemExit('No GPU — set Runtime → Change runtime type → GPU')


## 2. Install deps (zarr is not preinstalled on Colab)


In [ ]:
!pip -q install zarr kaggle 2>/dev/null
print('done')


## 3. Kaggle auth
Paste the contents of your `kaggle.json` (the token file), or just the token string.


In [ ]:
import os, json, getpass, pathlib
raw = getpass.getpass('Paste kaggle.json contents (or KGAT_ token): ').strip()
pathlib.Path('/root/.kaggle').mkdir(exist_ok=True)
if raw.startswith('{'):
    creds = json.loads(raw)
else:
    # bearer token form
    creds = {'username': 'colab', 'key': raw}
    os.environ['KAGGLE_API_TOKEN'] = raw
json.dump(creds, open('/root/.kaggle/kaggle.json', 'w'))
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle configured for user:', creds.get('username'))


## 4. Get the `ctrack` package (from the ctrack-src Kaggle dataset)


In [ ]:
import sys, shutil, pathlib, subprocess
subprocess.run(['kaggle','datasets','download','-d','maruanbottoni/ctrack-src',
                '-p','/content/ctrack_src','--unzip'], check=True)
# The dataset unzips the package CONTENTS to the root, so materialise a proper ctrack/ dir.
root = pathlib.Path('/content/ctrack_src')
pkg = pathlib.Path('/content/pkg/ctrack')
if pkg.exists(): shutil.rmtree('/content/pkg')
pkg.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(root, pkg)
sys.path.insert(0, '/content/pkg')
import ctrack; from ctrack.detect.unet3d import UNet3D; print('ctrack OK')


## 5. Download training data
Per-file download (no 87 GB zip). `N_VOLUMES` volumes + all ground-truth `.geff`.
199 volumes ≈ 87 GB; the default fits a Colab Pro disk with room to spare. Increase if
your runtime has the disk.


In [ ]:
N_VOLUMES = 150   # of 199; ~0.55 GB each

import urllib.request, urllib.parse, json, time, os, pathlib
COMP = 'biohub-cell-tracking-during-development'
TOK = os.environ.get('KAGGLE_API_TOKEN') or json.load(open('/root/.kaggle/kaggle.json')).get('key')
HDR = {'Authorization': 'Bearer ' + TOK, 'User-Agent': 'colab'}
DEST = pathlib.Path('/content/data/train'); DEST.mkdir(parents=True, exist_ok=True)

def api(url):
    return json.loads(urllib.request.urlopen(urllib.request.Request(url, headers=HDR), timeout=120).read())

# enumerate files (paged)
files, tok, page = [], None, 0
while True:
    u = f'https://www.kaggle.com/api/v1/competitions/data/list/{COMP}?pageSize=200'
    if tok: u += '&pageToken=' + urllib.parse.quote(tok)
    for a in range(6):
        try: d = api(u); break
        except Exception as e: time.sleep(5*(a+1))
    files += d.get('files', []); tok = d.get('nextPageTokenNullable'); page += 1
    if not tok or not d.get('files'): break
    time.sleep(1)
print('listed', len(files), 'files')

# pick N train volumes (by .zarr store) + all geff
import re, collections
stores = collections.defaultdict(list)
for f in files:
    m = re.match(r'train/([^/]+)\.(zarr|geff)/', f['name'])
    if m: stores[m.group(1)].append(f['name'])
chosen = sorted(stores)[:N_VOLUMES]
# stores[s] already holds both the .zarr and .geff files for each chosen volume
want = sorted({n for s in chosen for n in stores[s]})
print(f'downloading {len(chosen)} volumes, {len(want)} files')

def dl(name):
    dest = pathlib.Path('/content/data') / name
    if dest.exists(): return
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://www.kaggle.com/api/v1/competitions/data/download/{COMP}/' + urllib.parse.quote(name)
    for a in range(6):
        try:
            b = urllib.request.urlopen(urllib.request.Request(url, headers=HDR), timeout=300).read()
            dest.write_bytes(b); return
        except Exception: time.sleep(5*(a+1))

from concurrent.futures import ThreadPoolExecutor
t0 = time.time()
with ThreadPoolExecutor(max_workers=8) as ex:
    for i, _ in enumerate(ex.map(dl, want)):
        if (i+1) % 500 == 0: print(f'  {i+1}/{len(want)}  {time.time()-t0:.0f}s')
print('download done', f'{time.time()-t0:.0f}s')
RAW = GT = '/content/data/train'


## 6. Train (inline — same logic as scripts/train_detector.py, on GPU)


In [ ]:
FOLD, K, EPOCHS, BATCH, BASE, LR = 0, 5, 25, 8, 32, 1e-3
TRAIN_STRIDE, VAL_STRIDE = 2, 4

import numpy as np, time, torch
from torch.utils.data import DataLoader
from ctrack.data.geff import find_geff_stores
from ctrack.data.splits import hold_out_golden, kfold
from ctrack.data.volume import ISOTROPIC_SUBSAMPLE, isotropic_scale
from ctrack.detect.dataset import DetectionFrames
from ctrack.detect.unet3d import UNet3D, masked_bce_loss
from scipy import ndimage

device = torch.device('cuda')
all_ds = sorted(p.name.replace('.geff','') for p in find_geff_stores(GT))
# keep only datasets whose volume we actually downloaded
have = {p.name.replace('.zarr','') for p in pathlib.Path(RAW).glob('*.zarr')}
all_ds = [d for d in all_ds if d in have]
rem, golden = hold_out_golden(all_ds, n=8, seed=0)
split = kfold(rem, k=K, seed=0)[FOLD]
tr = [d for d in split.train if d in have]; va = [d for d in split.val if d in have]
print(f'fold {FOLD}: {len(tr)} train, {len(va)} val datasets')

train = DetectionFrames(tr, RAW, GT, augment=True, frame_stride=TRAIN_STRIDE)
val = DetectionFrames(va, RAW, GT, augment=False, frame_stride=VAL_STRIDE)
print(f'frames: {len(train)} train, {len(val)} val')
dl = DataLoader(train, batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)

model = UNet3D(base=BASE).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
scaler = torch.amp.GradScaler('cuda')
spacing = float(isotropic_scale(subsample=ISOTROPIC_SUBSAMPLE)[0])
nms = max(1, int(round(3.0/spacing)) | 1)

def peak_recall(hm, gt_iso, thr=0.5, r=7.0):
    if len(gt_iso) == 0: return 0, 0
    pooled = ndimage.maximum_filter(hm, size=nms, mode='nearest')
    pk = np.argwhere((hm == pooled) & (hm > thr)).astype(np.float64)
    if len(pk) == 0: return 0, len(gt_iso)
    d = np.linalg.norm((gt_iso[:,None,:]-pk[None,:,:])*spacing, axis=2)
    return int((d.min(1) <= r).sum()), len(gt_iso)

best = -1
for epoch in range(EPOCHS):
    model.train(); t0, tot = time.time(), 0.0
    for x, y, m in dl:
        x, y, m = x.to(device), y.to(device), m.to(device)
        opt.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = masked_bce_loss(model(x), y, m)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += float(loss.detach())
    sched.step()
    model.eval(); mt = tl = 0
    with torch.no_grad():
        for j in range(len(val)):
            ds, t = val.index[j]; fr = val.frame_only(j)
            xx = torch.from_numpy(np.ascontiguousarray(fr))[None,None].float().to(device)
            hm = torch.sigmoid(model(xx))[0,0].cpu().numpy()
            a, b = peak_recall(hm, val._centres[(ds,t)]/np.array(ISOTROPIC_SUBSAMPLE))
            mt += a; tl += b
    rec = mt/max(1,tl)
    print(f'epoch {epoch:2d} loss {tot/max(1,len(dl)):.4f} val peak-recall {rec:.3f} {time.time()-t0:.0f}s')
    if rec > best:
        best = rec
        torch.save({'model': model.state_dict(), 'base': BASE, 'fold': FOLD, 'recall': rec},
                   f'/content/unet_fold{FOLD}.pt')
print('best val peak-recall', best)


## 7. Push weights back to Kaggle
Creates/updates the `biohub-detector-weights` dataset, which the submission kernel attaches.


In [ ]:
import pathlib, json, subprocess
wdir = pathlib.Path('/content/weights_out'); wdir.mkdir(exist_ok=True)
for p in pathlib.Path('/content').glob('unet_fold*.pt'):
    (wdir / p.name).write_bytes(p.read_bytes())
meta = {'title': 'biohub-detector-weights', 'id': 'maruanbottoni/biohub-detector-weights',
        'licenses': [{'name': 'CC0-1.0'}]}
json.dump(meta, open(wdir/'dataset-metadata.json','w'))
exists = subprocess.run(['kaggle','datasets','status','maruanbottoni/biohub-detector-weights'],
                        capture_output=True, text=True).returncode == 0
cmd = (['kaggle','datasets','version','-p',str(wdir),'-m','colab weights','--dir-mode','zip']
       if exists else ['kaggle','datasets','create','-p',str(wdir),'--dir-mode','zip'])
print(subprocess.run(cmd, capture_output=True, text=True).stdout)
print('done — attach maruanbottoni/biohub-detector-weights to the inference kernel')
